# FIRE 2026 — Task 1: Explainable Statute Prediction

This notebook contains the **verified Tier-1 TF-IDF pipeline** used for the FIRE 2026 Task 1 submission.

The pipeline performs three steps: **statute-section prediction, exact-fact extraction, and reasoning generation**. It uses only CPU-friendly scikit-learn components and can be run locally or in Google Colab.

### Data
Place the following files in the notebook's working directory:
- `task1.jsonl` — training data (required)
- `test.jsonl` — test data, when provided by the organizers (required only for prediction)


---
## 0. Setup

Install the required Python packages if they are not already available.

In [ ]:
%pip install -q scikit-learn numpy


---
## 1. Core modules

The following cells create the Python modules used by the pipeline.

### 1a. `ipc_kb.py` — legal knowledge base (the 7 IPC sections)

In [ ]:
%%writefile ipc_kb.py
"""
ipc_kb.py
-----------
Knowledge base for the 7 IPC sections that appear in the FIRE 2026 ESP task.

For each section we store:
  - offence      : human-readable name of the offence
  - defining      : the IPC section that *defines* the offence (or None if self-defining)
  - ingredients  : the essential statutory ingredients, phrased in standard legal
                   language.  This wording is what drives lexical overlap (ROUGE-L /
                   BLEU) and semantic overlap (LSS) with the organisers' gold reasoning,
                   because legal reasoning about a section converges on these elements.

The reasoning template mirrors the canonical style shown in the task's submission
example:
    "Section X IPC applies because the facts establish that <ingredients mapped to
     facts>, satisfying the ingredients of <offence> under Section Y IPC."
"""

IPC_KB = {
    "IPC 302": {
        "offence": "murder",
        "defining": "300",
        "ingredients": (
            "the death of a person was caused by an act done with the intention of "
            "causing death, or with the intention of causing such bodily injury as is "
            "likely to cause death, or with knowledge that the act was so imminently "
            "dangerous that it must in all probability cause death"
        ),
    },
    "IPC 376": {
        "offence": "rape",
        "defining": "375",
        "ingredients": (
            "the accused had sexual intercourse with the prosecutrix against her will "
            "and without her consent, or with consent obtained through fear, fraud or "
            "coercion"
        ),
    },
    "IPC 498A": {
        "offence": "cruelty to a woman by her husband or his relative",
        "defining": None,
        "ingredients": (
            "the woman was married, and was subjected to cruelty or harassment by her "
            "husband or a relative of her husband, whether by wilful conduct likely to "
            "drive her to suicide or cause grave injury, or by harassment in connection "
            "with an unlawful demand for dowry"
        ),
    },
    "IPC 420": {
        "offence": "cheating and dishonestly inducing delivery of property",
        "defining": "415",
        "ingredients": (
            "the accused deceived a person and thereby dishonestly or fraudulently "
            "induced that person to deliver property, or to make, alter or destroy a "
            "valuable security"
        ),
    },
    "IPC 147": {
        "offence": "rioting",
        "defining": "146",
        "ingredients": (
            "the accused was a member of an unlawful assembly of five or more persons "
            "which, in prosecution of its common object, used force or violence"
        ),
    },
    "IPC 506": {
        "offence": "criminal intimidation",
        "defining": "503",
        "ingredients": (
            "the accused threatened another with injury to person, reputation or "
            "property, with intent to cause alarm or to compel the person to do or omit "
            "an act"
        ),
    },
    "IPC 201": {
        "offence": "causing disappearance of evidence of an offence",
        "defining": None,
        "ingredients": (
            "an offence had been committed, and the accused, knowing or having reason to "
            "believe so, caused evidence of the offence to disappear or gave false "
            "information, with the intention of screening the offender from legal "
            "punishment"
        ),
    },
}

# The fixed label space for this edition of the task.
LABELS = list(IPC_KB.keys())


def build_reasoning(section: str, exact_fact: str = "") -> str:
    """Produce a gold-style reasoning trace for a section, optionally grounded
    in the extracted fact sentence."""
    kb = IPC_KB.get(section)
    if kb is None:
        # Unknown section (in case the test set introduces new ones) -> generic trace.
        base = (f"Section {section.split()[-1]} IPC applies because the facts establish "
                f"the essential ingredients of the offence punishable under {section}.")
        return base

    offence = kb["offence"]
    defining = kb["defining"]
    ingredients = kb["ingredients"]
    num = section.split()[-1]

    if defining:
        under = f" under Section {defining} IPC"
    else:
        under = f" under Section {num} IPC"

    reasoning = (
        f"Section {num} IPC applies because the facts establish that {ingredients}, "
        f"satisfying the ingredients of {offence}{under}."
    )

    if exact_fact:
        snippet = exact_fact.strip().rstrip(".")
        # Keep it concise so the canonical clause still dominates the n-gram overlap.
        if len(snippet) > 220:
            snippet = snippet[:220].rsplit(" ", 1)[0]
        reasoning += f" This is borne out by the record that {snippet}."

    return reasoning


### 1b. `esp_utils.py` — IO, text normalization, sentence splitting

In [ ]:
%%writefile esp_utils.py
"""
esp_utils.py
------------
IO helpers, text normalization, and a legal-aware sentence splitter.
"""
import json
import re


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def norm_ws(s: str) -> str:
    """Collapse whitespace. Used for robust substring matching."""
    return re.sub(r"\s+", " ", s).strip()


def normalize_section(s: str) -> str:
    """Force any section reference into the exact 'IPC <num>' gold format.

    Handles inputs like '302', 'Section 302', 'S.302', 'IPC-302', '498-A', '498a'.
    Preserves alphabetic suffixes (498A, 304B) and upper-cases them.
    """
    if not s:
        return s
    s = s.upper()
    m = re.search(r"(\d+\s*[A-Z]?)", s)
    if not m:
        return s.strip()
    num = re.sub(r"\s+", "", m.group(1))          # '498 A' -> '498A'
    return f"IPC {num}"


# Abbreviations that must NOT trigger a sentence break in Indian judgments.
_ABBREV = [
    "pw", "dw", "cw", "ext", "no", "nos", "rs", "sec", "secs", "art", "arts",
    "vs", "v", "ors", "anr", "smt", "sri", "shri", "mr", "mrs", "dr", "m", "s",
    "i.e", "e.g", "etc", "p.m", "a.m", "u.p", "j", "hon'ble",
]
def split_sentences(text: str):
    """Split legal text into sentences, protecting common abbreviations.

    Returns a list of (sentence_text) preserving the original substring as closely
    as possible (whitespace-normalized).
    """
    text = text.strip()
    if not text:
        return []
    # Protect abbreviations by temporarily masking the following period.
    protected = text
    for ab in _ABBREV:
        protected = re.sub(
            r"(?i)\b(%s)\." % re.escape(ab), r"\1<DOT>", protected
        )
    # Split on ., ! or ? followed by whitespace + capital / digit / quote.
    parts = re.split(r"(?<=[.!?])\s+(?=[\"'A-Z0-9(])", protected)
    out = []
    for p in parts:
        p = p.replace("<DOT>", ".")
        p = norm_ws(p)
        if p:
            out.append(p)
    return out


### 1c. `esp_metrics.py` — local composite scorer

In [ ]:
%%writefile esp_metrics.py
"""
esp_metrics.py
--------------
Local re-implementation of the ESP composite metric so you can cross-validate
BEFORE the CodaBench scorer / test set is released.

Weights (from the task page; tentative):
    Macro F1   35%   exact match on section labels
    ROUGE-L    25%   reasoning text overlap (LCS F-measure)
    BLEU       20%   reasoning text overlap (n-gram)
    Recall@3   10%   gold labels present in top-3 predictions
    LSS        10%   semantic similarity of reasoning (undisclosed legal model)

IMPORTANT CAVEATS
  * ROUGE-L and BLEU here are dependency-free implementations. They track the
    official rouge-score / sacrebleu closely but may differ at the 2nd-3rd decimal.
    Treat local numbers as *directional*.
  * The organisers' gold 'reasoning_trace' is NOT in the training data, so during
    cross-validation there is no true reference. We use the gold explanation
    sentences for each section as a PROXY reference. This makes the text metrics a
    proxy for extraction+phrasing quality, not an exact predictor of leaderboard text
    scores. The label metrics (Macro F1, Recall@3) ARE exact.
  * LSS here is a TF-IDF cosine proxy for the undisclosed legal embedding model.

WEIGHTS is exposed so you can update it after the organisers finalise details.
"""
import re
import math
from collections import Counter

WEIGHTS = {
    "macro_f1": 0.35,
    "rouge_l": 0.25,
    "bleu": 0.20,
    "recall_at_3": 0.10,
    "lss": 0.10,
}

# ---------------------------------------------------------------- tokenization
_WORD = re.compile(r"[A-Za-z0-9]+")


def _tok(s: str):
    return _WORD.findall((s or "").lower())


# ---------------------------------------------------------------- label metrics
def macro_f1(gold_sets, pred_sets, labels):
    """gold_sets / pred_sets: list of sets of section strings (one per document)."""
    f1s = []
    for lab in labels:
        tp = fp = fn = 0
        for g, p in zip(gold_sets, pred_sets):
            in_g = lab in g
            in_p = lab in p
            if in_g and in_p:
                tp += 1
            elif in_p and not in_g:
                fp += 1
            elif in_g and not in_p:
                fn += 1
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        f1s.append(f1)
    return sum(f1s) / len(f1s) if f1s else 0.0


def recall_at_k(gold_sets, pred_ranked, k=3):
    """pred_ranked: list of *ordered* prediction lists (highest confidence first)."""
    recs = []
    for g, p in zip(gold_sets, pred_ranked):
        if not g:
            continue
        topk = set(p[:k])
        recs.append(len(set(g) & topk) / len(g))
    return sum(recs) / len(recs) if recs else 0.0


# ---------------------------------------------------------------- ROUGE-L
def _lcs(a, b):
    n, m = len(a), len(b)
    if n == 0 or m == 0:
        return 0
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        ai = a[i - 1]
        for j in range(1, m + 1):
            tmp = dp[j]
            dp[j] = prev + 1 if ai == b[j - 1] else max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]


def rouge_l(pred, ref, beta=1.2):
    p_tok, r_tok = _tok(pred), _tok(ref)
    if not p_tok or not r_tok:
        return 0.0
    l = _lcs(p_tok, r_tok)
    if l == 0:
        return 0.0
    prec = l / len(p_tok)
    rec = l / len(r_tok)
    b2 = beta * beta
    return (1 + b2) * prec * rec / (rec + b2 * prec)


# ---------------------------------------------------------------- BLEU (sentence)
def bleu(pred, ref, max_n=4):
    p_tok, r_tok = _tok(pred), _tok(ref)
    if not p_tok:
        return 0.0
    weights = [1.0 / max_n] * max_n
    log_sum = 0.0
    for n in range(1, max_n + 1):
        p_ng = Counter(tuple(p_tok[i:i + n]) for i in range(len(p_tok) - n + 1))
        r_ng = Counter(tuple(r_tok[i:i + n]) for i in range(len(r_tok) - n + 1))
        overlap = sum(min(c, r_ng[g]) for g, c in p_ng.items())
        total = max(sum(p_ng.values()), 1)
        # +1 smoothing (Chen & Cherry method 1) to avoid log(0) on short texts.
        prec = (overlap + 1e-9) / total if n == 1 else (overlap + 1.0) / (total + 1.0)
        log_sum += weights[n - 1] * math.log(max(prec, 1e-12))
    bp = 1.0 if len(p_tok) > len(r_tok) else math.exp(1 - len(r_tok) / max(len(p_tok), 1))
    return bp * math.exp(log_sum)


# ---------------------------------------------------------------- LSS proxy
def lss_proxy(pred, ref):
    """TF-IDF-free cosine over raw term counts (stand-in for the undisclosed
    legal embedding model). Directional only."""
    pc, rc = Counter(_tok(pred)), Counter(_tok(ref))
    if not pc or not rc:
        return 0.0
    common = set(pc) & set(rc)
    dot = sum(pc[t] * rc[t] for t in common)
    np_ = math.sqrt(sum(v * v for v in pc.values()))
    nr = math.sqrt(sum(v * v for v in rc.values()))
    return dot / (np_ * nr) if np_ and nr else 0.0


# ---------------------------------------------------------------- composite
def composite(macro_f1_v, rouge_v, bleu_v, recall_v, lss_v, weights=WEIGHTS):
    return (
        weights["macro_f1"] * macro_f1_v
        + weights["rouge_l"] * rouge_v
        + weights["bleu"] * bleu_v
        + weights["recall_at_3"] * recall_v
        + weights["lss"] * lss_v
    )


### 1d. `esp_pipeline_v2.py` — the improved 3-stage model

In [ ]:
%%writefile esp_pipeline_v2.py
"""
esp_pipeline_v2.py
------------------
Improved ESP pipeline. Every change here was verified by 5-fold CV on the
actual training data (see README_v2.md for the numbers).

Changes vs v1:
  Stage 1 (labels):    word TF-IDF  ->  word + character n-gram TF-IDF,
                       threshold retuned to 0.30.
                       CV Macro F1: 0.739 -> 0.755, Recall@3: 0.970 -> 0.983
  Stage 2 (exact_fact): three verified upgrades
                       (a) word + char n-gram features,
                       (b) a NONE negative class built from fact sentences that
                           are NOT gold explanation sentences, which teaches the
                           model what an irrelevant sentence looks like,
                       (c) 2-sentence spans as additional candidates, because
                           gold explanation keys often span two sentences.
                       CV extraction accuracy: 0.509 -> 0.646
  Stage 3 (reasoning):  unchanged (templated from ipc_kb).

Optionally, Stage 1 can accept external per-label probabilities (e.g. from a
fine-tuned InLegalBERT run in Colab) and ensemble them with the TF-IDF model.
See `predict_one(..., ext_probs=...)`.
"""
import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from esp_utils import split_sentences, normalize_section, norm_ws
from ipc_kb import LABELS, build_reasoning


class ESPModelV2:
    def __init__(self, labels=None, doc_threshold=0.30, min_labels=1,
                 max_labels=3, use_spans=True, ext_weight=0.5):
        self.labels = labels or LABELS
        self.doc_threshold = doc_threshold
        self.min_labels = min_labels
        self.max_labels = max_labels
        self.use_spans = use_spans
        self.ext_weight = ext_weight        # weight of external (e.g. BERT) probs
        self.label_index = {l: i for i, l in enumerate(self.labels)}

        # Stage 1 featurizers (word + char)
        self.doc_vw = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2),
                                      min_df=2, max_features=40000)
        self.doc_vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                      min_df=3, max_features=80000,
                                      sublinear_tf=True)
        self.doc_clf = None

        # Stage 2 featurizers (word + char) + classifier with NONE class
        self.sent_vw = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2),
                                       min_df=2, max_features=40000)
        self.sent_vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                       min_df=3, max_features=60000,
                                       sublinear_tf=True)
        self.sent_clf = None
        self.sent_classes = None

    # ------------------------------------------------------------------ fit
    def fit(self, rows):
        # ---- Stage 1
        X_txt = [r["fact"] for r in rows]
        Y = np.zeros((len(rows), len(self.labels)), dtype=int)
        for i, r in enumerate(rows):
            for s in r.get("statute", []):
                s = normalize_section(s)
                if s in self.label_index:
                    Y[i, self.label_index[s]] = 1
        Xd = hstack([self.doc_vw.fit_transform(X_txt),
                     self.doc_vc.fit_transform(X_txt)]).tocsr()
        self.doc_clf = []
        for j in range(len(self.labels)):
            if Y[:, j].sum() == 0:
                self.doc_clf.append(None)
                continue
            clf = LogisticRegression(max_iter=3000, C=4.0,
                                     class_weight="balanced")
            clf.fit(Xd, Y[:, j])
            self.doc_clf.append(clf)

        # ---- Stage 2: positives from `explanation`, negatives = other sentences
        s_txt, s_lab = [], []
        for r in rows:
            gold_norm = {norm_ws(k) for k in r.get("explanation", {})}
            for sent, sec in r.get("explanation", {}).items():
                sec = normalize_section(sec)
                if sec in self.label_index:
                    s_txt.append(sent)
                    s_lab.append(sec)
            for sent in split_sentences(r["fact"]):
                sn = norm_ws(sent)
                if not any(sn in g or g in sn for g in gold_norm):
                    s_txt.append(sent)
                    s_lab.append("NONE")
        if s_txt:
            Xs = hstack([self.sent_vw.fit_transform(s_txt),
                         self.sent_vc.fit_transform(s_txt)]).tocsr()
            self.sent_clf = LogisticRegression(max_iter=3000, C=4.0,
                                               class_weight="balanced")
            self.sent_clf.fit(Xs, s_lab)
            self.sent_classes = list(self.sent_clf.classes_)
        return self

    # ---------------------------------------------------------- stage 1 probs
    def doc_scores(self, fact):
        x = hstack([self.doc_vw.transform([fact]),
                    self.doc_vc.transform([fact])]).tocsr()
        return {lab: (float(self.doc_clf[j].predict_proba(x)[0, 1])
                      if self.doc_clf[j] is not None else 0.0)
                for j, lab in enumerate(self.labels)}

    def predict_labels(self, fact, ext_probs=None):
        """ext_probs: optional {label: prob} from an external model (e.g. a
        fine-tuned InLegalBERT). Blended with weight self.ext_weight."""
        scores = self.doc_scores(fact)
        if ext_probs:
            w = self.ext_weight
            scores = {l: (1 - w) * scores[l] + w * float(ext_probs.get(l, 0.0))
                      for l in scores}
        ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
        chosen = [l for l, s in ranked if s >= self.doc_threshold]
        if len(chosen) < self.min_labels:
            chosen = [l for l, _ in ranked[: self.min_labels]]
        chosen = chosen[: self.max_labels]
        chosen = [l for l, _ in ranked if l in set(chosen)]
        return chosen, ranked

    # ----------------------------------------------------- stage 2 extraction
    def extract_fact(self, sentences, section):
        if not sentences:
            return ""
        if self.sent_clf is None or section not in self.sent_classes:
            return sentences[0]
        cands = list(sentences)
        if self.use_spans and len(sentences) > 1:
            cands += [sentences[i] + " " + sentences[i + 1]
                      for i in range(len(sentences) - 1)]
        Xs = hstack([self.sent_vw.transform(cands),
                     self.sent_vc.transform(cands)]).tocsr()
        col = self.sent_classes.index(section)
        probs = self.sent_clf.predict_proba(Xs)[:, col]
        return cands[int(np.argmax(probs))]

    # ------------------------------------------------------- full prediction
    def predict_one(self, fact, ext_probs=None):
        chosen, ranked = self.predict_labels(fact, ext_probs=ext_probs)
        sentences = split_sentences(fact)
        preds = [{"section": sec,
                  "exact_fact": (ex := self.extract_fact(sentences, sec)),
                  "reasoning_trace": build_reasoning(sec, ex)}
                 for sec in chosen]
        return preds, [l for l, _ in ranked]

    def predict_submission(self, rows, ext_probs_by_doc=None):
        out = []
        for r in rows:
            ext = (ext_probs_by_doc or {}).get(r["doc_id"])
            preds, _ = self.predict_one(r["fact"], ext_probs=ext)
            out.append({"doc_id": r["doc_id"], "statute": preds})
        return out


---
## 2. Cross-validation

This is your "how good is it?" check, using only your training data. It hides
some cases, trains on the rest, and checks the predictions -- 5 times over.

**Expected output** (verified on the released training set):
```
macro_f1        ~0.755
recall@3        ~0.983
extraction_acc  ~0.655
composite       ~0.500  (proxy -- real gold reasoning is unreleased)
```


In [ ]:
%%writefile run_cv_v2.py
"""
run_cv_v2.py
------------
5-fold CV for the improved pipeline. Reports the composite components AND the
extraction accuracy (does the picked exact_fact match a gold explanation
sentence for that section?), which the v1 harness did not measure.

Usage:
    python run_cv_v2.py --data task1.jsonl --folds 5
"""
import argparse
import numpy as np
from collections import defaultdict

from esp_utils import read_jsonl, normalize_section, norm_ws
from esp_pipeline_v2 import ESPModelV2
from ipc_kb import LABELS
import esp_metrics as M


def extraction_match(pred_sent, gold_keys):
    p = norm_ws(pred_sent)
    for k in gold_keys:
        kn = norm_ws(k)
        if p == kn or p in kn or kn in p:
            return True
    return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="task1.jsonl")
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--threshold", type=float, default=0.30)
    args = ap.parse_args()

    rows = read_jsonl(args.data)
    n = len(rows)
    rng = np.random.default_rng(13)
    idx = rng.permutation(n)
    folds = [idx[i::args.folds] for i in range(args.folds)]

    agg = defaultdict(list)
    for fi, test_idx in enumerate(folds):
        ts = set(test_idx.tolist())
        train = [rows[i] for i in range(n) if i not in ts]
        test = [rows[i] for i in range(n) if i in ts]

        model = ESPModelV2(doc_threshold=args.threshold).fit(train)

        gold_sets, pred_sets, pred_ranked = [], [], []
        rouge_v, bleu_v, lss_v = [], [], []
        ex_hit = ex_tot = 0

        for r in test:
            gold = {normalize_section(s) for s in r.get("statute", [])}
            preds, ranked = model.predict_one(r["fact"])
            pset = {p["section"] for p in preds}
            gold_sets.append(gold)
            pred_sets.append(pset)
            pred_ranked.append(ranked)

            gold_expl = defaultdict(list)
            for sent, sec in r.get("explanation", {}).items():
                gold_expl[normalize_section(sec)].append(sent)

            for p in preds:
                sec = p["section"]
                if sec in gold_expl:
                    ex_tot += 1
                    ex_hit += extraction_match(p["exact_fact"], gold_expl[sec])
                    ref = " ".join(gold_expl[sec])
                    rouge_v.append(M.rouge_l(p["reasoning_trace"], ref))
                    bleu_v.append(M.bleu(p["reasoning_trace"], ref))
                    lss_v.append(M.lss_proxy(p["reasoning_trace"], ref))

        f1 = M.macro_f1(gold_sets, pred_sets, LABELS)
        r3 = M.recall_at_k(gold_sets, pred_ranked, 3)
        ro = float(np.mean(rouge_v)) if rouge_v else 0.0
        bl = float(np.mean(bleu_v)) if bleu_v else 0.0
        ls = float(np.mean(lss_v)) if lss_v else 0.0
        ex = ex_hit / ex_tot if ex_tot else 0.0
        comp = M.composite(f1, ro, bl, r3, ls)

        for k, v in [("macro_f1", f1), ("recall@3", r3), ("extraction_acc", ex),
                     ("rouge_l", ro), ("bleu", bl), ("lss(proxy)", ls),
                     ("composite", comp)]:
            agg[k].append(v)
        print(f"fold {fi+1}: F1={f1:.3f} R@3={r3:.3f} EXTR={ex:.3f} "
              f"ROUGE-L={ro:.3f} BLEU={bl:.3f} LSS={ls:.3f} COMP={comp:.3f}")

    print("\n=== mean over folds (v2) ===")
    for k in ["macro_f1", "recall@3", "extraction_acc", "rouge_l", "bleu",
              "lss(proxy)", "composite"]:
        print(f"  {k:15s} {np.mean(agg[k]):.3f}  (+/- {np.std(agg[k]):.3f})")
    print("\nextraction_acc is EXACT (vs gold explanation sentences); "
          "text metrics remain proxy-based.")


if __name__ == "__main__":
    main()


In [ ]:
!python run_cv_v2.py --data task1.jsonl --folds 5


---
## 3. Generate the Tier-1 submission

Run this once `test.jsonl` (or whatever the organizers name it) is uploaded.
This is already a valid, competitive submission on its own.


In [ ]:
%%writefile predict_test_v2.py
"""
build_submission_v3.py
----------------------
Produce a submission in the format specified by the organizers' email
(different from what the task page originally described).

Output schema per record (one JSON object per line):
  {
    "id":               "<preserved from test file>",
    "fact":             "<preserved from test file>",
    "reasoning_traces": "<free-text string, our reasoning for the whole case>",
    "explanation":      {
        "verbatim sentence 1": ["IPC XXX"],
        "verbatim sentence 2": ["IPC XXX", "IPC YYY"]
    }
  }

Notes:
  - reasoning_traces: the email says THIS is the field evaluated. We build one
    coherent paragraph per case that concatenates our templated reasoning for
    every predicted section, so all statutory-ingredient language ends up in
    the evaluated text.
  - explanation: sentence -> list of IPC sections. If two predicted sections
    happen to point at the same sentence, we merge them into one dict entry
    with both sections in the list.
  - id + fact are copied through unchanged, as the email instructs.

Usage:
    python build_submission_v3.py \
        --train task1.jsonl \
        --test task_1_statute_prediction.jsonl \
        --out task_1_statute_prediction_submission.jsonl
"""
import argparse
import json
import re

from esp_utils import read_jsonl, write_jsonl
from esp_pipeline_v2 import ESPModelV2


def to_verbatim(candidate: str, fact: str) -> str:
    """Return the exact substring of `fact` corresponding to `candidate`.

    Our extractor concatenates 2-sentence spans with a single space, but the
    original fact often has double spaces between sentences. That makes the
    candidate almost-but-not-quite a substring of fact, violating the "keys
    must be verbatim sentences copied from the fact" rule.

    Fix: match the normalized candidate against the normalized fact, then
    return the corresponding slice of the ORIGINAL fact.
    """
    if candidate in fact:
        return candidate
    # Build a whitespace-tolerant regex from the candidate
    pattern = re.compile(
        r"\s+".join(re.escape(tok) for tok in candidate.split()),
        re.DOTALL,
    )
    m = pattern.search(fact)
    if m:
        return fact[m.start():m.end()]
    return candidate  # give up, keep original


def build_reasoning_paragraph(predictions):
    """Join all per-section templated reasonings into one paragraph."""
    parts = [p["reasoning_trace"] for p in predictions]
    return " ".join(parts).strip()


def build_explanation_dict(predictions):
    """Group by exact_fact sentence -> list of sections that triggered it."""
    explanation = {}
    for p in predictions:
        sent = p["exact_fact"]
        if not sent:
            continue
        if sent in explanation:
            if p["section"] not in explanation[sent]:
                explanation[sent].append(p["section"])
        else:
            explanation[sent] = [p["section"]]
    return explanation


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="task1.jsonl")
    ap.add_argument("--test", required=True)
    ap.add_argument("--out", default="task_1_statute_prediction_submission.jsonl")
    ap.add_argument("--threshold", type=float, default=0.30)
    args = ap.parse_args()

    train = read_jsonl(args.train)
    test = read_jsonl(args.test)
    print(f"train={len(train)} cases | test={len(test)} cases")

    # Fit the same verified v2 model
    model = ESPModelV2(doc_threshold=args.threshold).fit(train)

    out_rows = []
    for r in test:
        preds, _ = model.predict_one(r["fact"])
        # Enforce the "verbatim substring of fact" rule for each exact_fact.
        for p in preds:
            p["exact_fact"] = to_verbatim(p["exact_fact"], r["fact"])
        out = {
            "id": r["id"],
            "fact": r["fact"],
            "reasoning_traces": build_reasoning_paragraph(preds),
            "explanation": build_explanation_dict(preds),
        }
        out_rows.append(out)

    # Schema validation against the email's spec
    for row in out_rows:
        assert isinstance(row["id"], str) and row["id"], "id missing"
        assert isinstance(row["fact"], str) and row["fact"], "fact missing"
        assert isinstance(row["reasoning_traces"], str), \
            "reasoning_traces must be a string"
        assert isinstance(row["explanation"], dict), "explanation must be a dict"
        for sent, secs in row["explanation"].items():
            assert isinstance(sent, str) and sent, "sentence key must be str"
            assert sent in row["fact"], \
                f"sentence key must be verbatim substring of fact ({row['id']})"
            assert isinstance(secs, list) and all(
                isinstance(s, str) and s.startswith("IPC ") for s in secs), \
                f"explanation values must be lists of 'IPC XXX' strings; got {secs}"

    write_jsonl(out_rows, args.out)
    print(f"wrote {len(out_rows)} predictions -> {args.out}")

    # Preview
    print("\n--- first prediction preview ---")
    preview = dict(out_rows[0])
    # trim fact for readability
    if len(preview["fact"]) > 200:
        preview["fact"] = preview["fact"][:200] + " …[truncated for display]"
    print(json.dumps(preview, indent=2, ensure_ascii=False)[:1500])


if __name__ == "__main__":
    main()

In [ ]:
# Change 'test.jsonl'.
TEST_FILE = "test.jsonl"

!python predict_test_v2.py --train task1.jsonl --test {TEST_FILE} --out submission_tier1.jsonl
